# MeetStream Feature Manual — Every Piece, Demoed Individually, With Real Data

This is the instruction manual *and* the test: every feature this repository ships — the
bridge server's raw HTTP contract and `langchain-meetstream` — demoed one at a time against
a **real** MeetStream meeting, with real credentials. For a leaner "just run the whole
pipeline" version, see
[`notebooks/end_to_end_test.ipynb`](end_to_end_test.ipynb). For the same material as static
prose, see [`docs/LANGCHAIN.md`](../docs/LANGCHAIN.md) and [`docs/BRIDGE_SERVER.md`](../docs/BRIDGE_SERVER.md).

**How to use this notebook**: it dispatches exactly **one** real bot (or reuses one you
already dispatched) and threads that same `bot_id` through every section below — you will
not join multiple meetings. Because a real meeting has a real timeline (you need to actually
be in it for chat/image actions to do anything, and a transcript only exists once MeetStream
finishes processing it), this is meant to be run cell by cell with real-world pauses in
between, not blindly "Run All" — the markdown before each cell tells you what to expect and
when to wait.

**Before running**: fill in every `<INSERT ... HERE>` value in section 2. Cells that need a
value you left blank raise, not silently skip.

## Contents

1. Setup — install the two local projects
2. Configuration — your real credentials and meeting info
3. Architecture recap — how the bridge and package fit together
4. The bridge server — the raw HTTP contract itself, no framework involved
5. `langchain-meetstream` — the client, the loader, all 6 tools individually, the agent, RAG, errors
6. The verification suite — what else this repo has, and where to find it
7. Cleanup


## 1. Setup

Installs the two independently-versioned local projects this repo contains (`bridge/`, and
`langchain_meetstream` at the repository root), each via `pip install -e <local path>` --
this repo has never been published to PyPI, and none of this needs it to be. See
[`docs/LANGCHAIN.md`](../docs/LANGCHAIN.md) if that looks surprising.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "docs" / "ARCHITECTURE.md").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError(
            "Could not locate the meetstream-langchain repo root. "
            "Run this notebook from inside the cloned repository."
        )
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts" / "verification"))
import _lib

print("Repo root:", REPO_ROOT)
print("Python:", sys.executable)


In [ ]:
!{sys.executable} -m pip install -e "{REPO_ROOT / 'bridge'}[dev]"
!{sys.executable} -m pip install -e "{REPO_ROOT}[dev,examples]"


In [ ]:
# Editable installs register their import hooks via a .pth file that Python's
# `site` module only processes at interpreter startup -- a package installed
# mid-session (like the one above) isn't importable in *this* kernel without
# either a restart or pointing sys.path directly at its source directory,
# which is what this cell does.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 2. Configuration — insert your real credentials here

Replace every `<INSERT ... HERE>` below. Nothing here is committed anywhere — this cell only
sets local Python variables for the rest of this notebook. **Never paste a real key into a
chat conversation** — type it directly into this cell in your own editor.

- `MEETSTREAM_API_KEY` — **required**. Sent to MeetStream as `Authorization: Token <key>`.
- `MEETING_URL` — a real, currently-joinable Google Meet / Zoom / Teams URL. This notebook
  dispatches a bot into it once.
- `EXISTING_BOT_ID` — alternative to `MEETING_URL`: reuse a `bot_id` from a bot you already
  dispatched (this notebook, `end_to_end_test.ipynb`, or `live_meetstream_test.py dispatch`).
  Fill in exactly one of `MEETING_URL` / `EXISTING_BOT_ID`. Note: the live-action demos in
  section 4 (chat message, image, leave) only work while a bot is *actively in* a meeting —
  reusing an old `bot_id` from an already-ended meeting means those specific cells are
  skipped (clearly marked below), while the read-only ones (status, transcript, tools) still
  run for real.
- `OPENAI_API_KEY` — needed only for the agent and RAG cells. Leave as the placeholder to
  skip just those; everything else still runs for real.
- `DEMO_CHAT_MESSAGE` / `DEMO_IMAGE_URL` — have working defaults, override if you like.


In [ ]:
MEETSTREAM_API_KEY = "<INSERT YOUR REAL MEETSTREAM API KEY HERE>"

# Fill in exactly one of these two:
MEETING_URL = "<INSERT A REAL MEETING URL HERE, e.g. https://meet.google.com/abc-defg-hij>"
EXISTING_BOT_ID = ""  # e.g. "bot_123" -- leave blank if using MEETING_URL instead

# Only needed for the agent/RAG cells -- leave as the placeholder to skip just those.
OPENAI_API_KEY = "<INSERT YOUR REAL OPENAI API KEY HERE>"

# Used by the live send_meeting_chat_message / send_meeting_image demos -- safe defaults.
DEMO_CHAT_MESSAGE = "Hello from the MeetStream feature-manual notebook!"
DEMO_IMAGE_URL = "https://placehold.co/600x400.png"


def _is_placeholder(value: str) -> bool:
    return not value or value.startswith("<INSERT")


if _is_placeholder(MEETSTREAM_API_KEY):
    raise ValueError("Set MEETSTREAM_API_KEY above to your real MeetStream API key before continuing.")

if _is_placeholder(MEETING_URL) and not EXISTING_BOT_ID:
    raise ValueError("Set MEETING_URL to a real meeting URL, or EXISTING_BOT_ID to an already-dispatched bot_id.")
if not _is_placeholder(MEETING_URL) and EXISTING_BOT_ID:
    raise ValueError("Set only one of MEETING_URL / EXISTING_BOT_ID, not both.")

FRESH_DISPATCH = not _is_placeholder(MEETING_URL)
if not FRESH_DISPATCH:
    MEETING_URL = ""  # normalize the unused placeholder to empty

RUN_LLM_CELLS = not _is_placeholder(OPENAI_API_KEY)
if RUN_LLM_CELLS:
    import os

    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # langchain_openai reads this from the environment
else:
    print("OPENAI_API_KEY left as placeholder -- agent and RAG cells below will be skipped.")

if not FRESH_DISPATCH:
    print(f"Reusing EXISTING_BOT_ID={EXISTING_BOT_ID!r} -- live action cells (chat/image/leave) will be skipped.")


## 3. Architecture recap

Two independently-versioned things, one direction of dependency:

```
Google Meet / Zoom / Teams
          |
      MeetStream  (external -- real API, real key, only the bridge talks to it)
          |
     Bridge Server  (bridge/ -- FastAPI, no LangChain dependency)
          |
   langchain-meetstream
          |
   Your agent / RAG app
```

- **The bridge** (`bridge/`) is the *only* thing that ever sees your real MeetStream API key
  or MeetStream's raw response shapes. It normalizes everything into one typed HTTP contract.
- **`langchain_meetstream`** talks only to the bridge, over plain HTTP — never to MeetStream
  directly. Section 5 below exposes four capabilities (dispatch, meeting status, transcript,
  live actions) in LangChain's native shape.

Full rationale: [`docs/ARCHITECTURE.md`](../docs/ARCHITECTURE.md).


## 4. The bridge server — the raw HTTP contract

This section talks to the bridge with plain `httpx`, not through either framework package,
so you can see the actual wire contract both packages are built on top of. Full endpoint
reference: [`docs/BRIDGE_SERVER.md`](../docs/BRIDGE_SERVER.md).


In [ ]:
import httpx

bridge = _lib.TemporaryBridge(api_key=MEETSTREAM_API_KEY)
bridge.__enter__()
print("Bridge running at", bridge.base_url)


### `GET /health`

Liveness check. Deliberately does not call MeetStream -- it reports whether the bridge
process itself is up, not whether MeetStream is reachable.


In [ ]:
response = httpx.get(f"{bridge.base_url}/health")
print(response.status_code, response.json())


### `POST /bots` — dispatch a bot into a meeting

`meeting_id` in every bridge response *is* MeetStream's `bot_id` -- MeetStream has no
meeting concept independent of the bot that joined it (`docs/ARCHITECTURE.md` §6). If you're
reusing `EXISTING_BOT_ID` instead, this cell skips the real call and just uses that value.


In [ ]:
if FRESH_DISPATCH:
    response = httpx.post(
        f"{bridge.base_url}/bots",
        json={"meeting_url": MEETING_URL, "bot_name": "Feature Manual Bot"},
    )
    print(response.status_code, response.json())
    bot_id = response.json()["bot_id"]
else:
    bot_id = EXISTING_BOT_ID
    print("Reusing existing bot_id:", bot_id)


### `GET /meetings/{meeting_id}` — status and metadata

`title`, `started_at`, `ended_at` are almost always `null` -- MeetStream's real API doesn't
expose them synchronously (`docs/ARCHITECTURE.md` §6), not a bug in the bridge.


In [ ]:
response = httpx.get(f"{bridge.base_url}/meetings/{bot_id}")
print(response.status_code, response.json())


### Aside: the error model

Every bridge error is `{"error": {"code", "message"}}` with a matching HTTP status. Here it
is for real, by asking about a meeting that doesn't exist:


In [ ]:
response = httpx.get(f"{bridge.base_url}/meetings/does-not-exist-12345")
print(response.status_code, response.json())


### `POST /meetings/{meeting_id}/actions` — live meeting actions

Three confirmed-real actions: `send_chat_message`, `send_image`, `leave_meeting`. These only
do something while the bot is *actively in* the meeting -- skipped below if you're reusing
`EXISTING_BOT_ID` from an already-ended meeting. If this is a fresh dispatch, make sure the
bot has actually joined (check the status above) before running these.


In [ ]:
if FRESH_DISPATCH:
    response = httpx.post(
        f"{bridge.base_url}/meetings/{bot_id}/actions",
        json={"action": "send_chat_message", "payload": {"message": DEMO_CHAT_MESSAGE}},
    )
    print(response.status_code, response.json())
else:
    print("Skipping -- reusing an existing bot_id, not actively in a meeting.")


In [ ]:
if FRESH_DISPATCH:
    response = httpx.post(
        f"{bridge.base_url}/meetings/{bot_id}/actions",
        json={"action": "send_image", "payload": {"image_url": DEMO_IMAGE_URL, "display_duration_seconds": 10}},
    )
    print(response.status_code, response.json())
else:
    print("Skipping -- reusing an existing bot_id, not actively in a meeting.")


### Leaving the meeting

Run this cell once you're done interacting with the live meeting -- it makes the bot leave.
Recorded data (the transcript) is preserved and still fetchable afterward.


In [ ]:
if FRESH_DISPATCH:
    response = httpx.post(f"{bridge.base_url}/meetings/{bot_id}/actions", json={"action": "leave_meeting"})
    print(response.status_code, response.json())
else:
    print("Skipping -- reusing an existing bot_id that has already left.")


### `GET /meetings/{meeting_id}/transcript`

Transcription is asynchronous on MeetStream's side -- this polls, treating HTTP 409
(`transcript_not_ready`) as "try again shortly." If MeetStream reports something other than
"still processing" (e.g. a job that hasn't started yet), re-run this cell after a bit rather
than assuming failure -- see `docs/ARCHITECTURE.md` §7.


In [ ]:
import time


def wait_for_transcript_raw(base_url, meeting_id, attempts=10, delay_seconds=15):
    for attempt in range(attempts):
        response = httpx.get(f"{base_url}/meetings/{meeting_id}/transcript")
        if response.status_code == 200:
            return response.json()
        if response.status_code == 409:
            print(f"Not ready yet ({attempt + 1}/{attempts}) -- waiting {delay_seconds}s...")
            time.sleep(delay_seconds)
            continue
        raise RuntimeError(f"Unexpected response {response.status_code}: {response.json()}")
    raise TimeoutError(f"Transcript for {meeting_id} still not ready after {attempts} attempts")


raw_transcript = wait_for_transcript_raw(bridge.base_url, bot_id)
print(f"{len(raw_transcript['segments'])} real segment(s)")
for segment in raw_transcript["segments"][:3]:
    print(segment)


## 5. `langchain-meetstream`

Same `bot_id` as above, now through the LangChain-native package. Full guide:
[`docs/LANGCHAIN.md`](../docs/LANGCHAIN.md).

### The client

`MeetStreamClient` is a synchronous, connection-pooled `httpx.Client` that talks only to the
bridge (never to MeetStream directly). It's shared internally between the loader and every
tool below -- one instance, reused.


In [ ]:
from langchain_meetstream import MeetStreamAPIError, MeetStreamClient, MeetStreamLoader
from langchain_meetstream.tools import get_meetstream_tools

lc_client = MeetStreamClient(api_key=MEETSTREAM_API_KEY, base_url=bridge.base_url)
print("Client constructed, pointed at", bridge.base_url)


### `MeetStreamLoader` — transcript to `Document`s

One `Document` per speaker turn (not one blob per meeting) -- so retrieval can return the
exact turn relevant to a question, with `speaker`/`start_time` attached for citation.


In [ ]:
lc_docs = MeetStreamLoader(meeting_id=bot_id, client=lc_client).load()
print(f"{len(lc_docs)} Document(s)")
for doc in lc_docs[:3]:
    print(doc.page_content)
    print(doc.metadata)
    print()


### `get_meetstream_tools` — all 6 tools, at a glance


In [ ]:
lc_tools = get_meetstream_tools(lc_client)
lc_tools_by_name = {t.name: t for t in lc_tools}
for tool in lc_tools:
    summary = (tool.description or "").splitlines()[0] if tool.description else "(no description)"
    print(f"{tool.name}: {summary}")


### Tool 1/6 — `dispatch_meetstream_bot`

Args: `meeting_url`, `bot_name`. Already exercised for real via the raw bridge call in
section 4 -- shown here as its schema only, not re-invoked (invoking again would dispatch a
second real bot into a meeting).


In [ ]:
tool = lc_tools_by_name["dispatch_meetstream_bot"]
print(tool.description)
print("args:", tool.args)


### Tool 2/6 — `get_meeting`

Args: `meeting_id`. Read-only, safe to call any time -- invoked for real below.


In [ ]:
tool = lc_tools_by_name["get_meeting"]
print(tool.invoke({"meeting_id": bot_id}))


### Tool 3/6 — `get_meeting_transcript`

Args: `meeting_id`. Read-only, safe to call any time -- invoked for real below.


In [ ]:
tool = lc_tools_by_name["get_meeting_transcript"]
result = tool.invoke({"meeting_id": bot_id})
print(f"{len(result['segments'])} segment(s)")


### Tool 4/6 — `send_meeting_chat_message`

Args: `meeting_id`, `message`. Already exercised for real via the raw bridge call in
section 4 -- schema only here, not re-invoked (would post a duplicate real chat message).


In [ ]:
tool = lc_tools_by_name["send_meeting_chat_message"]
print(tool.description)
print("args:", tool.args)


### Tool 5/6 — `send_meeting_image`

Args: `meeting_id`, `image_url`, `display_duration_seconds`. Already exercised for real via
the raw bridge call in section 4 -- schema only here.


In [ ]:
tool = lc_tools_by_name["send_meeting_image"]
print(tool.description)
print("args:", tool.args)


### Tool 6/6 — `leave_meeting`

Args: `meeting_id`. Already exercised for real via the raw bridge call in section 4 --
schema only here (invoking again on an already-left bot would just 404).


In [ ]:
tool = lc_tools_by_name["leave_meeting"]
print(tool.description)
print("args:", tool.args)


### A real LangChain agent

Same check as `scripts/verification/live_langchain_agent.py`: does a real tool-calling model
decide on its own to call `get_meeting_transcript`. Skipped if `OPENAI_API_KEY` was left as
the placeholder in section 2.


In [ ]:
if RUN_LLM_CELLS:
    from langchain.agents import create_agent
    from langchain_openai import ChatOpenAI

    lc_agent = create_agent(model=ChatOpenAI(model="gpt-4o-mini"), tools=lc_tools)
    lc_result = lc_agent.invoke(
        {"messages": [{"role": "user", "content": f"What did people say in meeting {bot_id}? Give me a short summary."}]}
    )
    print(lc_result["messages"][-1].content)
else:
    print("Skipping -- OPENAI_API_KEY not set in section 2.")


### RAG over the real transcript

Minimal inline version of `examples/rag_example.py` -- reuses `lc_docs` already loaded above.


In [ ]:
if RUN_LLM_CELLS:
    from langchain_core.vectorstores import InMemoryVectorStore
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(lc_docs)
    store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings())

    question = "What was discussed in this meeting?"
    context = "\n\n".join(d.page_content for d in store.similarity_search(question, k=4))
    answer = ChatOpenAI(model="gpt-4o-mini").invoke(f"Answer using only this context:\n{context}\n\nQuestion: {question}")
    print(answer.content)
else:
    print("Skipping -- OPENAI_API_KEY not set in section 2.")


### Error handling

Every call through this package raises one exception type, `MeetStreamAPIError`, with a
`.code` matching the bridge's error table -- regardless of which tool/loader call triggered
it. Demonstrated here for real, same as the bridge-level error demo in section 4:


In [ ]:
try:
    lc_client.get_meeting("does-not-exist-12345")
except MeetStreamAPIError as exc:
    print(f"code={exc.code!r} status={exc.status_code} message={str(exc)!r}")


## 6. The verification suite

Beyond the package demoed above, `scripts/verification/` is a separate demo-readiness /
smoke-test suite: mocked structural checks that need no real MeetStream account
(`check_bridge.py`, `check_langchain.py`, `check_packaging.py`, `check_tests.py`), plus
real, credentialed scripts mirroring what this notebook just did by hand
(`live_meetstream_test.py`, `live_langchain_test.py` / `_tools.py` / `_agent.py`). It's
intentionally not re-run inside this notebook -- running it is one command:

```bash
python scripts/verification/verify_all.py
```

Full catalog and reasoning: [`scripts/verification/README.md`](../scripts/verification/README.md).
Pre-demo checklist: [`scripts/verification/DEMO_CHECKLIST.md`](../scripts/verification/DEMO_CHECKLIST.md).


## 7. Cleanup


In [ ]:
bridge.__exit__(None, None, None)
print("Bridge stopped.")
